<a href="https://colab.research.google.com/github/eviemcmahan/walmart-hurricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Hurricane Analytics

**DS2002 Midterm Project | Fall 2026 | University of Virginia**

**Team:** HEJ

**Members:** Evie McMahan, Julia McPhillips, Honor Dold

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Data Cleaning

Cleaning functions, the SKU mapping table, and the decision log.

Graded on a clear pipeline: duplicates, missing values, SKU consolidation, type fixes.

Done when one function turns raw files into clean tables. The log lists every decision and rows touched.

**Data problems:**
- Duplicate records (DONE)
- Missing values written several ways: empty, NULL, N/A, NaN (DONE)
- Dates and times in different formats (DONE)
- The same product under several SKUs and names (DONE)
- Inconsistent capitalization (DONE) and spacing (DONE)
- Category names that don't match (DONE)
- Numbers stored as text, prices with $ signs (DONE)
- Negative quantities (DONE)
- Store ids written different ways across files (DONE)
- Weather data in a different timezone than sales (DONE on the sales side; the weather side is applied in Section 2)

### Profile the data

In [27]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd, os

def find_data(filename, folders=('data', '../data', '/kaggle/input', '/content')):
    for folder in folders:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return None

def profile(df, name='dataset'):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    print()
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print()
    print('exact duplicate rows:', df.duplicated().sum())
    return summary

After profiling the walmart_transactions data, I need to pull in other data files to compare these values to what the real counts are.

In [28]:
files = { # Group all the other files together to loop over
    'transactions': 'walmart_transactions.csv',
    'stores': 'store_locations.csv',
    'catalog': 'product_catalog.csv',
    'distribution_centers': 'distribution_centers.csv',

}

raw = {}
for name, filename in files.items(): # Loop through each file
    file_path = find_data(filename)
    if file_path: # Use the code that was given in lecture to read the file and profile it if found
        raw[name] = pd.read_csv(file_path)
        profile(raw[name], name)
        print()
    else:
        print(f'Could not find {filename}. Upload the data/ folder in Kaggle or Colab, or run this inside the repo.')

tx_raw = raw['transactions']

=== transactions ===
25,562 rows x 8 columns

                 dtype  nulls  null_pct  distinct
transaction_id  object      0       0.0     25532
store_id        object      0       0.0        60
timestamp       object    886       3.5     23952
sku             object      0       0.0        45
product_name    object      0       0.0        63
category        object      0       0.0        37
quantity         int64      0       0.0        11
unit_price      object      0       0.0      3668

exact duplicate rows: 30

=== stores ===
14 rows x 8 columns

              dtype  nulls  null_pct  distinct
store_id     object      0       0.0        14
store_name   object      0       0.0        12
city         object      0       0.0        12
state        object      0       0.0         3
zip_code      int64      0       0.0        12
latitude    float64      1       7.1        11
longitude   float64      1       7.1        11
region       object      0       0.0         7

exact duplicate r

In [29]:
def spelling_noise(df, cols): # Check for basic spelling and formatting noise
    for c in cols: # Loop through columns
        raw = df[c].dropna().astype(str)
        norm = raw.str.strip().str.lower().str.replace(r'[^a-z0-9]', '', regex=True) # Normalize values
        print(f'{c:15s} raw distinct: {raw.nunique():3d} | after normalizing: {norm.nunique():3d}') # Compare raw distinct values to normalized values

spelling_noise(raw['transactions'], ['store_id', 'product_name', 'category', 'sku'])
spelling_noise(raw['stores'], ['store_id', 'state', 'city'])
spelling_noise(raw['catalog'], ['product_name', 'category', 'subcategory'])

store_id        raw distinct:  60 | after normalizing:  12
product_name    raw distinct:  63 | after normalizing:  49
category        raw distinct:  37 | after normalizing:  18
sku             raw distinct:  45 | after normalizing:  45
store_id        raw distinct:  14 | after normalizing:  12
state           raw distinct:   3 | after normalizing:   3
city            raw distinct:  12 | after normalizing:  12
product_name    raw distinct:  38 | after normalizing:  34
category        raw distinct:  27 | after normalizing:  15
subcategory     raw distinct:  19 | after normalizing:  19


In [30]:
# Look closer at whichever columns worried you most
tx = tx_raw.copy()

if 'transactions' in raw:
    print('sample of sku values:')
    print(tx['sku'].value_counts().head(10))
    print()
    print('sample of raw quantity values:')
    print(tx['quantity'].astype(str).value_counts().head(10))

sample of sku values:
sku
TARP-HD         766
LED-FLASH       732
FLASH-01        728
HDTARP-01       701
CHRG-PORT       684
CANDLE-EMRG     664
CNDL-EM         652
BEER-24         643
FIRSTAID-KIT    637
PORTCHARGER     632
Name: count, dtype: int64

sample of raw quantity values:
quantity
1     10268
2      9290
3      3745
4      1343
5       528
6       258
-1       54
-2       53
-3       15
-5        4
Name: count, dtype: int64


The store profile shows that there are 14 rows in the store file, but there are 60 distinct `store_id`s in the transactions profile. This means there is spelling and formatting noise, since there should only be 12 distinct `store_id`s (the store file has 14 rows, but 2 are duplicates).

The catalog profile shows that there are 45 distinct SKUs, which matches the 45 distinct `sku`s in transactions. Catalog also shows that there are 38 distinct product names, but there are 63 distinct `product_name` values in transactions. Finally, there are 27 distinct categories in catalog, but there are 37 in the transactions data.

So far, within the transactions dataset I need to remove the noise in `store_id`, `product_name`, and `category`. After cleaning, `store_id` should have 12 distinct values (the store file has 14 rows, but 2 are duplicates). The `sku` column is clean (45 valid codes), but the catalog has only 38 product names for 45 SKUs, so several SKUs are the same product.

We will also have to do formatting normalizing for some of the rows based on the basic normalizing check.

On data types, we will have to address `unit_price`, which is currently a string, along with some others. `quantity` also has negative values. We also have to address the null values that are shown in the profile.

### Decision Log

In [31]:
decision_log = [] # Create decision log

def log(step, column, decision, rows_before=None, rows_after=None):
    decision_log.append({
        'step': step,
        'column': column,
        'decision': decision,
        'rows_before': rows_before,
        'rows_after': rows_after,
    })

# After each cleaning step, call log(...).
# View the whole log with: pd.DataFrame(decision_log)

### #1: Trim inconsistent spacing

In [32]:
def trim_text(df, name):
    """Trim and collapse spaces in every string columns; blank strings become missing."""
    out = df.copy()
    text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
    changed = {}
    for c in text_cols: # Go through the cleaning process
        s = out[c]
        cleaned = s.str.strip().str.replace(r'\s+', ' ', regex=True).replace('', np.nan)
        changed[c] = int((s.fillna('') != cleaned.fillna('')).sum())
        out[c] = cleaned
    fixed = {k: v for k, v in changed.items() if v}
    log(f'{name}: trim spaces', ', '.join(text_cols), # Log the decision
        f'trimmed/collapsed spaces, blanks set to missing; values changed: {fixed}',
        len(df), len(out))
    return out

cleaned = {}
for name, df in raw.items(): # Trim each string column in each dataframe dataframe
    cleaned[name] = trim_text(df, name)

pd.DataFrame(decision_log)

,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5


Trimming the duplicate spacing in every row in a string column impacted only `store_id`, `product_name`, `unit_price`, and `region`. All rows were preserved.

### #2: Deal with missing values: empty, NULL, N/A, NaN

In [33]:
missing_types = ['', 'NULL', 'N/A', 'NaN', 'nan'] # Different types of missing values

def check_missing(path):
    text = pd.read_csv(path, dtype=str, keep_default_na=False)   # Read as plain text
    for col in text.columns: # Loop through each column
        values = text[col].str.strip() # Remove spaces
        missing = values[values.isin(missing_types)] # Find missing values
        if len(missing) > 0: # If there are missing values
            print(f'  {col}: {len(missing)} missing, written as:',
                  missing.replace('', '(blank)').value_counts().to_dict())

for name, filename in files.items():
    print(name)
    check_missing(find_data(filename))

transactions
  timestamp: 886 missing, written as: {'NULL': 188, 'N/A': 181, '(blank)': 176, 'NaN': 173, 'nan': 168}
stores
  latitude: 1 missing, written as: {'(blank)': 1}
  longitude: 1 missing, written as: {'(blank)': 1}
catalog
  subcategory: 13 missing, written as: {'(blank)': 13}
  unit_cost: 5 missing, written as: {'(blank)': 5}
distribution_centers
  cost_per_shipment: 1 missing, written as: {'(blank)': 1}


In [34]:
missing = cleaned['transactions']['timestamp'].isna()
print(f'{missing.sum()} of {len(cleaned['transactions']):,} rows missing a timestamp ({missing.mean():.1%})')

stores = cleaned['stores']
stores[stores['latitude'].isna() | stores['longitude'].isna()] # Determine which store is missing lat and long

886 of 25,562 rows missing a timestamp (3.5%)


,store_id,store_name,city,state,zip_code,latitude,longitude,region
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle


When deciding what to do with missing values, I split it up between `timestamp` and the other missing values, since there are a much greater number of missing values in `timestamp` (886 vs the 21 other combined values). I left most of the other missing values in place because there were so few and those columns are not used in the analysis. The exceptions are the 13 missing catalog subcategories, which I filled with "unknown", and Tallahassee's missing latitude and longitude, which I will fill in by lookup since Q4 needs coordinates. This will just be a city-level approximation for latitude and longitude, not the actual store location.

886 timestamps in transactions (3.5%) are still missing a timestamp. I decided to keep these rows with the timestamp left missing instead of dropping them or filling in a date. Dropping them would delete real sales from the store and category totals, and any date I filled in would be a guess that could put sales on the wrong day. Since the date is the only thing missing, I will exclude these rows only from analysis that needs a date (Q1 and Q3) and keep them everywhere else.

In [35]:
def fill_missing(tables):
    out = dict(tables)

    # Catalog: subcategory is a text column, so fill it with a label
    c = out['catalog'].copy()
    n = int(c['subcategory'].isna().sum())
    c['subcategory'] = c['subcategory'].fillna('unknown')
    log('catalog: missing values', 'subcategory', f'filled {n} missing values with "unknown"', len(c), len(c))
    log('catalog: missing values', 'unit_cost', f'left {int(c["unit_cost"].isna().sum())} missing; not used in Q1 to Q5', len(c), len(c))
    out['catalog'] = c

    # Stores: Tallahassee's missing coordinates, city level, from latlong.net
    s = out['stores'].copy()
    mask = s['latitude'].isna() | s['longitude'].isna()
    s.loc[mask, ['latitude', 'longitude']] = [30.455, -84.2533]
    log('stores: missing values', 'latitude, longitude',
        f'filled {int(mask.sum())} store ({s.loc[mask, "city"].tolist()[0]}) with city-level coordinates from latlong.net', len(s), len(s))
    out['stores'] = s

    # Distribution centers: leave Birmingham's shipping cost missing
    d = out['distribution_centers'].copy()
    log('distribution centers: missing values', 'cost_per_shipment',
        f'left {int(d["cost_per_shipment"].isna().sum())} missing (Birmingham); exclude from any cost comparison', len(d), len(d))
    out['distribution_centers'] = d

    # Transactions: timestamp is left missing on purpose; just log the decision
    t = out['transactions']
    log('transactions: missing values', 'timestamp',
        f'kept {int(t["timestamp"].isna().sum())} rows with missing timestamp; excluded only from analysis that needs a date', len(t), len(t))
    return out

cleaned = fill_missing(cleaned)
pd.DataFrame(decision_log)

,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562


### #3: Drop Duplicates

Our profiling work showed that transactions has 30 duplicates, but the other DataFrames don't have any duplicates. I will now drop the duplicates in transactions.

In [36]:
def drop_duplicates(df, name): # Drop duplicates function
    n = len(df)
    out = df.drop_duplicates() # Drop the duplicates
    dropped = n - len(out) # Count how many duplicates were dropped
    log(f'{name}: duplicates', 'all columns',
        f'dropped {dropped} exact duplicate rows (kept first copy)', n, len(out)) # Decision log
    print(f'{name}: {n:,} rows to {len(out):,} rows ({dropped} dropped)')
    return out

cleaned['transactions'] = drop_duplicates(cleaned['transactions'], 'transactions') # Run the function
pd.DataFrame(decision_log)


transactions: 25,562 rows to 25,532 rows (30 dropped)


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


### #4: DataType Formatting

Based on my profiling above, I saw that I need to clean the datatypes for `unit_price` and `timestamps` in transactions and `surge_capacity` in distribution centers.

In [37]:
def to_price(s):
    """'$3.38' -> 3.38"""
    out = pd.to_numeric(s.str.replace(r'[$,]', '', regex=True), errors='coerce')
    assert out.notna().all(), 'some prices failed to convert'
    return out

def to_datetime(s):
    """Parse all formats; values that can't be parsed become missing."""
    return pd.to_datetime(s, errors='coerce', format='mixed')

def to_capacity(s):
    """'~750' -> 750"""
    return pd.to_numeric(s.astype(str).str.replace(r'[^0-9.]', '', regex=True))

def fix_types(tables): # Fix all the types
    out = dict(tables)

    t = out['transactions'].copy()
    t['unit_price'] = to_price(t['unit_price']) # Fix unit price in transactions
    t['timestamp'] = to_datetime(t['timestamp']) # Fix timestamp in transactions
    log('transactions: types', 'unit_price, timestamp',
        'unit_price to float (removed $); timestamp parsed to datetime; missing timestamps left missing',
        len(t), len(t))
    out['transactions'] = t

    d = out['distribution_centers'].copy()
    d['surge_capacity_approx'] = d['surge_capacity'].astype(str).str.contains('~')
    d['surge_capacity'] = to_capacity(d['surge_capacity']) # Fix surge capacity in distribution centers
    log('distribution centers: types', 'surge_capacity', 'removed ~, converted to number; approximate value flagged', len(d), len(d))
    out['distribution_centers'] = d
    return out

cleaned = fix_types(cleaned) # Clean types on trimmed dataframes
cleaned['transactions'].dtypes
pd.DataFrame(decision_log)


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


After building functions to convert the three columns to their correct units, I made one overarching `fix_types` function to pass through my entire set of DataFrames.

### #5: Dealing with quantity

In [38]:
neg = cleaned['transactions'][cleaned['transactions']['quantity'] < 0]
print(f"{len(neg)} negative rows, {neg['quantity'].sum()} units (out of {cleaned['transactions']['quantity'].sum():,})")
neg.head()

130 negative rows, -241 units (out of 49,343)


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
163,TXN008747,FL-078,2024-09-22 21:30:35,PAPERTOWEL6,Paper Towels,HOUSEHOLD,-2,7.91
223,TXN006255,FL-619,2024-09-20 17:10:14,DIAPERS-3,Diapers Size 3,baby,-2,25.79
563,TXN013118,FL-330,2024-09-25 11:32:05,GAME-MONO,Board Game - Monopoly,ENTERTAINMENT,-2,19.58
811,TXN022136,FL-287,2024-09-06 13:20:05,TP-12,TP 12 Pack,household,-2,9.64
1034,TXN018762,FL-619,2024-09-30 14:42:48,FAID-01,First-Aid Kit,Health,-2,14.45


In [39]:
t = cleaned['transactions'] # Check if there are returns (same transaction_id and opposite quantity)
neg = t[t['quantity'] < 0]
print(neg['transaction_id'].isin(t.loc[t['quantity'] > 0, 'transaction_id']).sum())

0


I saw quantity had some negative values in the transactions DataFrame, which shouldn't be the case. After counting them, I found 130 negative rows that represent a quantity of -241 (out of a total quantity of 49,343 units, which includes the -241 subtracted). I can't leave them negative, since that would subtract values from the total quantity count. I decided to flip the values because they are all small, they are not matched by a positive row with the same transaction_id, so they don't look like returns, and they look like sign typos.

In [40]:
def flip_negative_quantity(df, name):
    out = df.copy()
    n_neg = int((out['quantity'] < 0).sum()) # Determine the negative rows first
    out['quantity'] = out['quantity'].abs() # Flip the sign
    log(f'{name}: quantity', 'quantity', # Log the decision
        f'flipped sign on {n_neg} negative rows (likely sign typos)', len(df), len(out))
    print(f'{name}: flipped {n_neg} negative quantities, total units from {df["quantity"].sum():,} to {out["quantity"].sum():,}')
    return out

cleaned['transactions'] = flip_negative_quantity(cleaned['transactions'], 'transactions') # Run the function
print('min quantity now:', cleaned['transactions']['quantity'].min()) # Check the minimum quanitity
pd.DataFrame(decision_log)

transactions: flipped 130 negative quantities, total units from 49,343 to 49,825
min quantity now: 1


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


### #6: Store ids written different ways across files

The profile showed 60 distinct `store_id` values in transactions after trimming spaces, but there are only 12 stores. The same store is written four ways (`FL-239`, `FL239`, `FL_239`, `fl-239`). The store file has the same problem: `FL239` and `FL_287` are extra copies of Orlando and Kissimmee, which is why it has 14 rows for 12 stores. The SQLite database already uses the `FL-239` format, so I will normalize everything to that.

In [41]:
def normalize_store_id(s):
    """'fl_239', 'FL239', 'FL 239' -> 'FL-239'"""
    s = s.str.upper().str.strip().str.replace(r'[^A-Z0-9]', '', regex=True) # Uppercase and remove separators
    return s.str[:2] + '-' + s.str[2:] # Put the dash back after the state letters

def fix_store_ids(tables):
    out = dict(tables)
    for name in ['transactions', 'stores']: # Both files have the store_id column
        df = out[name].copy()
        before = df['store_id'].nunique()
        new_ids = normalize_store_id(df['store_id'])
        changed = int((df['store_id'] != new_ids).sum())
        df['store_id'] = new_ids
        log(f'{name}: store ids', 'store_id', # Log the decision
            f'normalized to FL-### format; {changed} values changed; distinct ids {before} -> {df["store_id"].nunique()}',
            len(df), len(df))
        print(f'{name}: distinct store ids {before} -> {df["store_id"].nunique()} ({changed} values changed)')
        out[name] = df
    return out

cleaned = fix_store_ids(cleaned) # Run the function
cleaned['stores'][cleaned['stores']['store_id'].duplicated(keep=False)] # The store rows that are now duplicates

transactions: distinct store ids 48 -> 12 (3607 values changed)
stores: distinct store ids 14 -> 12 (2 values changed)


,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central
12,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
13,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central


After normalizing, the store file has two `store_id`s that appear twice. The duplicate rows match their originals in every other column, so these are the same store entered twice and not two different stores. This finishes the duplicates step from #3: I drop duplicates on `store_id` in the store file now that the ids match. If I left them in, every Orlando and Kissimmee sale would be counted twice when we join transactions to stores for Q4.

In [42]:
def drop_duplicate_stores(df):
    n = len(df)
    out = df.drop_duplicates(subset='store_id') # Keep the first copy of each store
    log('stores: duplicates', 'store_id',
        f'dropped {n - len(out)} duplicate stores after normalizing store_id (kept first copy)', n, len(out))
    print(f'stores: {n} rows to {len(out)} rows ({n - len(out)} dropped)')
    return out

cleaned['stores'] = drop_duplicate_stores(cleaned['stores']) # Run the function
assert cleaned['stores']['store_id'].is_unique

# Check that every transaction now finds its store, without duplicating any rows
t = cleaned['transactions']
check = t.merge(cleaned['stores'][['store_id', 'city']], on='store_id', how='left', validate='many_to_one')
assert len(check) == len(t), 'the join changed the row count'
assert check['city'].notna().all(), 'some store ids did not match the store file'
print('every transaction matches one store:', len(check) == len(t))
pd.DataFrame(decision_log)

stores: 14 rows to 12 rows (2 dropped)
every transaction matches one store: True


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


All 48 spellings in transactions collapsed to the 12 real stores, and the store file is down to 12 rows. The join check uses `validate='many_to_one'`, so if a duplicate store ever comes back the merge will raise an error instead of silently doubling sales. Every transaction matched a store and the row count did not change.

### #7: Inconsistent capitalization

`category` is written in every capitalization (`EMERGENCY`, `Emergency`, `emergency`), and `state` in the store file is written as `FL`, `Florida`, and `Fla.`. I will strip and title-case the category columns in transactions and the catalog, and change the state names to the `FL` abbreviation. Product names also have mixed capitalization, but title-casing them would not be enough on its own (`Pop Tarts - Strawberry` and `Pop-Tarts Strawberry` would still be different), so those are handled by the SKU mapping in #9.

In [43]:
def fix_capitalization(tables):
    out = dict(tables)

    for name in ['transactions', 'catalog']: # Both files have a category column
        df = out[name].copy()
        before = df['category'].nunique()
        df['category'] = df['category'].str.strip().str.title() # Title-case the category
        log(f'{name}: capitalization', 'category',
            f'title-cased category; distinct values {before} -> {df["category"].nunique()}', len(df), len(df))
        print(f'{name}: distinct categories {before} -> {df["category"].nunique()}')
        out[name] = df

    s = out['stores'].copy()
    state_fix = {'Florida': 'FL', 'Fla.': 'FL'} # State written three ways
    n = int(s['state'].isin(state_fix.keys()).sum())
    s['state'] = s['state'].replace(state_fix)
    log('stores: capitalization', 'state', f'changed {n} values (Florida, Fla.) to FL', len(s), len(s))
    print(f'stores: changed {n} state values to FL')
    out['stores'] = s
    return out

cleaned = fix_capitalization(cleaned) # Run the function
print()
print(cleaned['transactions']['category'].value_counts().sort_index()) # Distinct categories after title-casing

transactions: distinct categories 37 -> 18
catalog: distinct categories 27 -> 15
stores: changed 3 state values to FL

category
Alcohol                   864
Alcoholic Beverages       401
Baby                     1589
Baby Products             497
Breakfast                1569
Breakfast Foods           493
Electronics              2522
Emergency                5371
Emergency Supplies       1819
Entertainment            1893
Games & Entertainment     645
Groceries                 858
Grocery                  2451
Health                    937
Health & Wellness         294
Household                1779
Household Essentials      571
Snacks                    979
Name: count, dtype: int64


Title-casing took transactions from 37 categories down to 18 and the catalog from 27 down to 15. Every store is now `FL`. But some of the remaining categories are still two labels for one thing: `Breakfast` and `Breakfast Foods`, `Emergency` and `Emergency Supplies`, `Grocery` and `Groceries`, and so on. Those are fixed in the next step.

### #8: Category names that don't match

After fixing capitalization there are 18 categories, but the SQLite database (`daily_sales_summary`) uses exactly 10: Alcohol, Baby, Breakfast, Electronics, Emergency, Entertainment, Grocery, Health, Household, and Snacks. I mapped each leftover label onto one of those 10, so our transactions will line up with the database in Q5. Any category that is not in the map is kept as itself, and the code prints it so we can see if we missed one.

In [44]:
CATEGORY_MAP = {
    'Breakfast Foods': 'Breakfast',
    'Emergency Supplies': 'Emergency',
    'Alcoholic Beverages': 'Alcohol',
    'Games & Entertainment': 'Entertainment',
    'Groceries': 'Grocery',
    'Health & Wellness': 'Health',
    'Household Essentials': 'Household',
    'Baby Products': 'Baby',
}
DB_CATEGORIES = {'Alcohol', 'Baby', 'Breakfast', 'Electronics', 'Emergency',
                 'Entertainment', 'Grocery', 'Health', 'Household', 'Snacks'}

def fix_categories(tables):
    out = dict(tables)
    for name in ['transactions', 'catalog']:
        df = out[name].copy()
        before = df['category'].nunique()
        n = int(df['category'].isin(CATEGORY_MAP.keys()).sum())
        df['category'] = df['category'].replace(CATEGORY_MAP) # Unmapped labels stay as themselves
        leftover = set(df['category']) - DB_CATEGORIES
        print(f'{name}: distinct categories {before} -> {df["category"].nunique()}, not in the database list: {leftover or "none"}')
        log(f'{name}: category names', 'category',
            f'mapped {n} rows to the 10 database category names; distinct {before} -> {df["category"].nunique()}',
            len(df), len(df))
        out[name] = df
    return out

cleaned = fix_categories(cleaned) # Run the function

# Each SKU should now have exactly one category
cats_per_sku = cleaned['transactions'].groupby('sku')['category'].nunique()
print('SKUs with more than one category:', (cats_per_sku > 1).sum())
pd.DataFrame(decision_log)

transactions: distinct categories 18 -> 10, not in the database list: none
catalog: distinct categories 15 -> 10, not in the database list: none
SKUs with more than one category: 0


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


Both files now use the same 10 categories as the database, and every SKU has exactly one category (before this step each SKU had 3 or 4). This matters for Q1, since a category split across `Emergency` and `Emergency Supplies` would show two smaller surges instead of one big one.

One thing to note for Q5: `Games & Entertainment` was mapped to `Entertainment`, so Monopoly and playing cards are in the same category, just like in the database. That is the "category lumping" Q5 asks about, so for Q5 we should look at the SKU and product level instead of the category total.

### #9: The same product under several SKUs and names

The catalog has 45 SKUs but only 38 product names, and transactions has 62 different product names. The SKU codes themselves are clean (all 45 transaction SKUs are in the catalog), but the same product shows up under 2 to 4 different SKUs. Following the workflow from lecture, I first used code to build a candidate list: lowercase each name, remove punctuation, and sort the words, then group by that and look for groups with more than one SKU.

In [45]:
cat = cleaned['catalog'].copy()
cat['normalized'] = (cat['product_name'].str.lower()
                     .str.replace(r'[^a-z ]', '', regex=True) # Remove punctuation and numbers
                     .str.split().apply(lambda w: ' '.join(sorted(w)))) # Sort the words

candidates = cat.groupby('normalized')['sku'].agg(list).loc[lambda s: s.str.len() > 1]
print(f'{len(candidates)} candidate groups found by code:')
candidates

13 candidate groups found by code:


,sku
normalized,
aa batteries pack,"[BAT-AA4, BATT_AA_4PK]"
beer light pack,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candles emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"
cards playing,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,"[CHIPS-VP, CHIPVARIETY]"


The code found 13 groups, which covers most of the catalog, but it missed products whose names differ in small ways: `Strawberry Pop-Tart` vs `Pop-Tarts Strawberry` (`tart` vs `tarts`), `BATTERIES AA 4PK` vs `AA Batteries 4-Pack` (`pk` vs `pack`), `First-Aid` vs `First Aid`, `Infant Formula` vs `Baby Formula`, and the water, charger, toilet paper, diaper, and streaming SKUs. The code narrowed the list down, but I had to make the rest of the calls myself by looking at the catalog. I put the decisions in a mapping table instead of a dictionary, so it can be reviewed and edited, and so a left join shows any SKU we have not mapped yet. For each product I picked one of its existing SKUs as the canonical SKU.

`supplier` and `package_size` were no help in matching, because they are noisy too (light beer is listed as supplied by Hasbro). The judgment calls I was least sure about:
- **Roku Streaming Stick (`STREAM-01`) and Streaming Device (`ROKU-STD`):** merged. The `ROKU-STD` code and the transaction names (`STREAMING STICK`) point to the same Roku product.
- **Monopoly and playing cards:** kept separate. They are different products, and Q5 needs to tell them apart.
- **Pop-Tarts:** all four SKUs are strawberry, so they are one product. There are no other flavors in the data.

In [46]:
sku_map = pd.DataFrame([
    # (raw_sku, canonical_sku, product)
    ('PT-12', 'POPTART-STRAW', 'Strawberry Pop-Tarts'),
    ('SKU#459812', 'POPTART-STRAW', 'Strawberry Pop-Tarts'),
    ('POPTART-STRAW', 'POPTART-STRAW', 'Strawberry Pop-Tarts'),
    ('PT_STRAWBERRY', 'POPTART-STRAW', 'Strawberry Pop-Tarts'),
    ('WTR-24', 'WATER24PK', 'Bottled Water 24-Pack'),
    ('WATER24PK', 'WATER24PK', 'Bottled Water 24-Pack'),
    ('WTR_24PK', 'WATER24PK', 'Bottled Water 24-Pack'),
    ('BATT-100', 'BAT-AA4', 'AA Batteries 4-Pack'),
    ('BAT-AA4', 'BAT-AA4', 'AA Batteries 4-Pack'),
    ('BATT_AA_4PK', 'BAT-AA4', 'AA Batteries 4-Pack'),
    ('FLASH-01', 'LED-FLASH', 'LED Flashlight'),
    ('LED-FLASH', 'LED-FLASH', 'LED Flashlight'),
    ('BEER-24', 'BEER-24', 'Light Beer 24-Pack'),
    ('LTBEER24', 'BEER-24', 'Light Beer 24-Pack'),
    ('BG-MONO', 'MONOPOLY-01', 'Monopoly Board Game'),
    ('MONOPOLY-01', 'MONOPOLY-01', 'Monopoly Board Game'),
    ('GAME-MONO', 'MONOPOLY-01', 'Monopoly Board Game'),
    ('CARDS-01', 'PLAYCARD-STD', 'Playing Cards'),
    ('PLAYCARD-STD', 'PLAYCARD-STD', 'Playing Cards'),
    ('STREAM-01', 'ROKU-STD', 'Roku Streaming Stick'),
    ('ROKU-STD', 'ROKU-STD', 'Roku Streaming Stick'),
    ('SOUP-CAN', 'SOUP-CAN', 'Canned Soup'),
    ('CNSOUP-01', 'SOUP-CAN', 'Canned Soup'),
    ('BREAD-WH', 'BREAD-WH', 'White Bread'),
    ('WHITEBREAD', 'BREAD-WH', 'White Bread'),
    ('PB-16', 'PB-16', 'Peanut Butter 16oz'),
    ('PNUTBTR-16', 'PB-16', 'Peanut Butter 16oz'),
    ('CNDL-EM', 'CANDLE-EMRG', 'Emergency Candles'),
    ('CANDLE-EMRG', 'CANDLE-EMRG', 'Emergency Candles'),
    ('FAID-01', 'FIRSTAID-KIT', 'First Aid Kit'),
    ('FIRSTAID-KIT', 'FIRSTAID-KIT', 'First Aid Kit'),
    ('CHRG-PORT', 'PORTCHARGER', 'Portable Phone Charger'),
    ('PORTCHARGER', 'PORTCHARGER', 'Portable Phone Charger'),
    ('CHIPS-VP', 'CHIPVARIETY', 'Chips Variety Pack'),
    ('CHIPVARIETY', 'CHIPVARIETY', 'Chips Variety Pack'),
    ('PTOWEL-6', 'PAPERTOWEL6', 'Paper Towels 6-Roll'),
    ('PAPERTOWEL6', 'PAPERTOWEL6', 'Paper Towels 6-Roll'),
    ('TP-12', 'TOILETPAPER12', 'Toilet Paper 12-Roll'),
    ('TOILETPAPER12', 'TOILETPAPER12', 'Toilet Paper 12-Roll'),
    ('DIAP-S3', 'DIAPERS-3', 'Diapers Size 3'),
    ('DIAPERS-3', 'DIAPERS-3', 'Diapers Size 3'),
    ('BFORM-01', 'BABYFORMULA', 'Baby Formula'),
    ('BABYFORMULA', 'BABYFORMULA', 'Baby Formula'),
    ('TARP-HD', 'TARP-HD', 'Heavy Duty Tarp'),
    ('HDTARP-01', 'TARP-HD', 'Heavy Duty Tarp'),
], columns=['raw_sku', 'canonical_sku', 'product'])

assert sku_map['raw_sku'].is_unique, 'a raw SKU is mapped twice'
print(f'{len(sku_map)} SKUs -> {sku_map["canonical_sku"].nunique()} canonical products')

45 SKUs -> 20 canonical products


Now I apply the mapping with a left join (so an unmapped SKU stays as itself instead of turning into `NaN`) and check that it is safe: no rows added or removed, no missing canonical SKUs, and the unit total unchanged. I keep the original `sku` and `product_name` columns and add `canonical_sku` and `product` next to them, because Q2 needs the fragmented view to compare against the consolidated one.

In [47]:
def consolidate_skus(df, name, sku_map):
    n = len(df)
    out = df.merge(sku_map, left_on='sku', right_on='raw_sku', how='left', validate='many_to_one')

    unmapped = sorted(out.loc[out['raw_sku'].isna(), 'sku'].unique()) # SKUs we have not mapped yet
    print(f'{name}: not yet in the mapping table: {unmapped}')
    out['canonical_sku'] = out['canonical_sku'].fillna(out['sku']) # Unmapped SKUs keep their own code
    out['product'] = out['product'].fillna(out['product_name'])
    out = out.drop(columns='raw_sku')

    assert len(out) == n, 'mapping changed the row count' # Consolidation renames, it never adds or removes
    assert out['canonical_sku'].notna().all(), 'mapping produced missing SKUs'
    if 'quantity' in out:
        assert out['quantity'].sum() == df['quantity'].sum(), 'mapping changed unit totals'

    log(f'{name}: SKU consolidation', 'sku, product_name',
        f'mapped {df["sku"].nunique()} SKUs to {out["canonical_sku"].nunique()} products with the mapping table; '
        f'added canonical_sku and product, kept the original columns; unmapped: {len(unmapped)}', n, len(out))
    print(f'{name}: {df["sku"].nunique()} SKUs -> {out["canonical_sku"].nunique()} products, rows {n:,} -> {len(out):,}')
    return out

cleaned['transactions'] = consolidate_skus(cleaned['transactions'], 'transactions', sku_map) # Run the function
cleaned['catalog'] = consolidate_skus(cleaned['catalog'], 'catalog', sku_map)

# Each product should now have one category
print('products with more than one category:',
      (cleaned['transactions'].groupby('product')['category'].nunique() > 1).sum())
pd.DataFrame(decision_log)

transactions: not yet in the mapping table: []
transactions: 45 SKUs -> 20 products, rows 25,532 -> 25,532
catalog: not yet in the mapping table: []
catalog: 45 SKUs -> 20 products, rows 45 -> 45
products with more than one category: 0


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


In [48]:
# Pop-Tarts before and after: the fragmented view vs the consolidated view
t = cleaned['transactions']
pt = t[t['canonical_sku'] == 'POPTART-STRAW']
print('Fragmented view (units by original SKU):')
print(pt.groupby('sku')['quantity'].sum().sort_values(ascending=False))
print()
print('Consolidated:', pt['quantity'].sum(), 'units')
print(f"The largest single SKU showed {pt.groupby('sku')['quantity'].sum().max()} units, "
      f"so the real number is {pt['quantity'].sum() / pt.groupby('sku')['quantity'].sum().max():.1f}x bigger than it looked")

Fragmented view (units by original SKU):
sku
PT-12            1385
POPTART-STRAW    1306
PT_STRAWBERRY    1270
SKU#459812       1257
Name: quantity, dtype: int64

Consolidated: 5218 units
The largest single SKU showed 1385 units, so the real number is 3.8x bigger than it looked


All 45 SKUs were in the mapping table (the unmapped list is empty), and they now map to 20 products. The row count and unit total did not change, and every product has a single category. The Pop-Tarts check shows why this matters: split across four SKUs, no single code looks like much, but together they are several times larger. This is the starting point for Q2.

Heads-up for Q5: the SQLite `inventory_levels` table has four SKUs that are not in the catalog or the transactions (`LEGACY-PT`, `MISC-001`, `OLD-BATT-50`, `UNKNOWN-99`). Those are worth checking when we audit the database.

### #10: Weather data in a different timezone than sales

The sales timestamps are in local Florida time (Eastern), and weather APIs often return times in UTC (the case says the weather feed is in GMT). Florida in September is on daylight time, which is UTC-4. If we joined without converting, any weather reading from 8 PM to midnight local time would land on the next day. That matters most on landfall night: Helene made landfall late on Sept 26, which is already Sept 27 in UTC.

On the sales side, I add a `date` column, which is the timestamp cut back to midnight with `.dt.normalize()`, so it can be merged with daily weather. Rows with a missing timestamp get a missing date, same as the decision in #2. For the weather side, I wrote a function that converts UTC times to Eastern time before taking the date. If Section 2 uses Open-Meteo with `timezone='America/New_York'` like the lecture recipe, the API already returns Eastern days, and the function is only needed for hourly or UTC data.

In [49]:
LOCAL_TZ = 'America/New_York'

def add_sales_date(df, name):
    out = df.copy()
    out['date'] = out['timestamp'].dt.normalize() # Timestamp cut back to midnight, local Eastern time
    log(f'{name}: timezone', 'timestamp -> date',
        f'sales timestamps treated as local Eastern time; added date column for the weather join; '
        f'{int(out["date"].isna().sum())} rows have no date (missing timestamp)', len(df), len(out))
    print(f'{name}: added date column, {out["date"].min():%Y-%m-%d} to {out["date"].max():%Y-%m-%d}, '
          f'{out["date"].nunique()} days')
    return out

def weather_to_local(weather, time_col='time', tz=LOCAL_TZ):
    """Convert UTC weather times to Eastern time and add a date column that matches the sales date."""
    out = weather.copy()
    t = pd.to_datetime(out[time_col], utc=True) # Treat the API times as UTC
    out['local_time'] = t.dt.tz_convert(tz).dt.tz_localize(None) # Convert to Eastern, drop the tz label
    out['date'] = out['local_time'].dt.normalize()
    return out

cleaned['transactions'] = add_sales_date(cleaned['transactions'], 'transactions') # Run the function

# Example with two made-up readings to show the conversion (not real weather data)
example = pd.DataFrame({'time': ['2024-09-26T20:00', '2024-09-27T03:00']}) # UTC
print(weather_to_local(example))
pd.DataFrame(decision_log)

transactions: added date column, 2024-09-01 to 2024-09-30, 30 days
               time          local_time       date
0  2024-09-26T20:00 2024-09-26 16:00:00 2024-09-26
1  2024-09-27T03:00 2024-09-26 23:00:00 2024-09-26


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


The example shows the problem: a reading at 3 AM UTC on Sept 27 is really 11 PM on Sept 26 in Florida, which is landfall night. Without converting, it would be counted on the wrong day. The transactions now cover Sept 1 to Sept 30 with a `date` column that is ready to merge with the daily weather in Section 2.

### Full cleaning pipeline

The section is done when one function turns the raw files into clean tables. `clean_all` runs every step above in order on the raw data, and resets the decision log first so it is not filled twice. The row counts at the end are the numbers to check if anything changes later.

In [50]:
def clean_all(raw):
    decision_log.clear() # Start a fresh log so each run gives the same result
    tables = {name: trim_text(df, name) for name, df in raw.items()} # 1
    tables = fill_missing(tables)                                     # 2
    tables['transactions'] = drop_duplicates(tables['transactions'], 'transactions') # 3
    tables = fix_types(tables)                                        # 4
    tables['transactions'] = flip_negative_quantity(tables['transactions'], 'transactions') # 5
    tables = fix_store_ids(tables)                                    # 6
    tables['stores'] = drop_duplicate_stores(tables['stores'])
    tables = fix_capitalization(tables)                               # 7
    tables = fix_categories(tables)                                   # 8
    tables['transactions'] = consolidate_skus(tables['transactions'], 'transactions', sku_map) # 9
    tables['catalog'] = consolidate_skus(tables['catalog'], 'catalog', sku_map)
    tables['transactions'] = add_sales_date(tables['transactions'], 'transactions') # 10
    return tables

clean = clean_all(raw)
print()
for name, df in clean.items():
    print(f'{name}: {len(raw[name]):,} raw rows -> {len(df):,} clean rows')
assert clean['transactions'].equals(cleaned['transactions']), 'the pipeline does not match the step-by-step result'
print('decisions logged:', len(decision_log))
pd.DataFrame(decision_log)

transactions: 25,562 rows to 25,532 rows (30 dropped)
transactions: flipped 130 negative quantities, total units from 49,343 to 49,825
transactions: distinct store ids 48 -> 12 (3607 values changed)
stores: distinct store ids 14 -> 12 (2 values changed)
stores: 14 rows to 12 rows (2 dropped)
transactions: distinct categories 37 -> 18
catalog: distinct categories 27 -> 15
stores: changed 3 state values to FL
transactions: distinct categories 18 -> 10, not in the database list: none
catalog: distinct categories 15 -> 10, not in the database list: none
transactions: not yet in the mapping table: []
transactions: 45 SKUs -> 20 products, rows 25,532 -> 25,532
catalog: not yet in the mapping table: []
catalog: 45 SKUs -> 20 products, rows 45 -> 45
transactions: added date column, 2024-09-01 to 2024-09-30, 30 days

transactions: 25,562 raw rows -> 25,532 clean rows
stores: 14 raw rows -> 12 clean rows
catalog: 45 raw rows -> 45 clean rows
distribution_centers: 5 raw rows -> 5 clean rows
decis

,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, prod...","trimmed/collapsed spaces, blanks set to missin...",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missin...",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supp...","trimmed/collapsed spaces, blanks set to missin...",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missin...",5,5
4,catalog: missing values,subcategory,"filled 13 missing values with ""unknown""",45,45
5,catalog: missing values,unit_cost,left 5 missing; not used in Q1 to Q5,45,45
6,stores: missing values,"latitude, longitude",filled 1 store (Tallahassee) with city-level c...,14,14
7,distribution centers: missing values,cost_per_shipment,left 1 missing (Birmingham); exclude from any ...,5,5
8,transactions: missing values,timestamp,kept 886 rows with missing timestamp; excluded...,25562,25562
9,transactions: duplicates,all columns,dropped 30 exact duplicate rows (kept first copy),25562,25532


## 2. Weather API

API call with retries, parsing, timezone alignment, and a checked join to sales.

Graded on a working call, timezones handled, and a join you checked.

Done when one function returns daily weather. The join passes a row-count check. Base URL cited.

## 3. The Five Questions

The five answers: code, a number, and a written finding each. Q5 uses SQLite.

Graded on a code, a visualization, and a written explanation for each one.

Done when every question has code, a number, and a paragraph.

## 4. Visualizations

At least six plots with message titles, the six-minute talk, and the README.

Graded on at least six plots with clear titles, labels, and legends.

Done when six labeled plots. The talk has been rehearsed once.

## 5. Reflection

Graded on five questions, one solid paragraph each, every member contributes.